# On-sky utilization: reproduction of the original "Visit Timeline" figure

This notebook reproduces, as faithfully as possible, the **Visit Timeline** figure from
`notebooks/on-sky_utilization.ipynb` in [lsst/sp_notebooks](https://github.com/lsst/sp_notebooks)
(commit `66e1baa8f5de348cda2a63fda00122b1c03f6da6`, plot logic by Keith Bechtol, SP-3121).
It is the **reference** for the interactive version in `on_sky_utilization_interactive.ipynb`.

Rules for this notebook:

- Cells marked **ORIGINAL** are copied verbatim from the source notebook, quirks included
  (BLOCK-T539 visits drawn twice, y-axis referenced to 00:00 UTC, unused start/end arrays).
- Only three things differ: the ConsDB connection cell (works both in a Nublado kernel and on
  an SDF login node), the date range, and where the figure is saved.
- The other two sections of the original (visit acquisition rate vs ideal night, and visit gap
  times) are out of scope here and are not reproduced.

See `../README.md` for the full description and `../ai_notes.md` for the verification log.

In [ ]:
# Parameters cell (MODIFIED: date range for the JOR 2026 / RASAB figure)
day_obs_min = 20251026  # Start of Early Operations period
day_obs_max = 20260714  # Last night requested for the reproduction

not_times_square = True
USE_CACHE = True  # read ../cache/visit1_<min>_<max>.parquet if it exists

In [ ]:
# ORIGINAL
SAVE_FIGURES = False
# SURVEY_PROGRAMS = ['BLOCK-365'] # 'BLOCK-407', 'BLOCK-408', 'BLOCK-416', 'BLOCK-417', 'BLOCK-419', 'BLOCK-421'] # Commissioning
SURVEY_PROGRAMS = [
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
]  # Early Ops
FOCUS_ALIGNMENT_PROGRAMS = ["BLOCK-T539"]

In [ ]:
# MODIFIED: save the figure next to this folder's plots/ instead of figures/
from pathlib import Path

SAVE_FIGURES = True
PLOTS_DIR = Path("../plots")
CACHE_DIR = Path("../cache")
PLOTS_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

In [ ]:
# ORIGINAL
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from astropy.time import Time
import astropy.units as u

from astroplan import Observer

# %matplotlib widget

In [ ]:
# ORIGINAL
from lsst.utils.plotting import publication_plots

publication_plots.set_rubin_plotstyle()

## ConsDB connection (modified)

The original calls `rubin_nights.connections.get_clients()`, which needs a Rubin Science
Platform (RSP) token. That works in a Nublado kernel. On an SDF login node (for example VS Code
over SSH) there is no token, so we fall back to the direct Postgres route that `rubin_nights`
also provides (`ConsDbSql`, USDF logical replica, credentials in
`~/.lsst/postgres-credentials.txt`). Both return the same `visit1` rows.

In [ ]:
# MODIFIED: token route first, Postgres replica fallback
import os
import logging

from rubin_nights import connections
from rubin_nights.consdb_query import ConsDbSql


def get_consdb_client():
    logging.disable(logging.ERROR)  # silence the expected "No RSP token found"
    try:
        client = connections.get_clients()["consdb"]
        if len(client.query("SELECT 1 AS one")) == 1:
            print("ConsDB via RSP token:", client)
            return client
    except Exception as err:  # noqa: BLE001
        print("Token route failed:", type(err).__name__)
    finally:
        logging.disable(logging.NOTSET)
    os.environ.setdefault(
        "PGPASSFILE", os.path.expanduser("~/.lsst/postgres-credentials.txt")
    )
    client = ConsDbSql(site="usdf")
    print("ConsDB via Postgres replica:", client)
    return client


client = get_consdb_client()

In [ ]:
# ORIGINAL
instrument = "lsstcam"

In [ ]:
# ORIGINAL (query text unchanged; the .sort_values matches the rubin_nights version)
def fetch(day_obs_min, day_obs_max):
    visits_query = f"""
    SELECT 
        *
    FROM
        cdb_{instrument}.visit1 AS visit1 
    WHERE
        airmass > 0.
        AND img_type in ('acq', 'science', 'cwfs', 'focus')
        AND day_obs >= {day_obs_min} AND day_obs <= {day_obs_max}
    """

    visits = client.query(visits_query).sort_values(by="obs_start_mjd")

    return visits

In [ ]:
# MODIFIED: parquet cache so the notebook can be re-run without ConsDB
cache_file = CACHE_DIR / f"visit1_{day_obs_min}_{day_obs_max}.parquet"
if USE_CACHE and cache_file.exists():
    visits = pd.read_parquet(cache_file)
    print("read", len(visits), "visits from", cache_file)
else:
    visits = fetch(day_obs_min, day_obs_max)
    visits.to_parquet(cache_file)
    print("fetched", len(visits), "visits, cached to", cache_file)

In [ ]:
# ORIGINAL
selection = np.isin(
    visits["science_program"], SURVEY_PROGRAMS
)  # & (visits['day_obs'] == 20250415)
print(visits["day_obs"][selection].value_counts().sort_index().to_string())
# visits[selection]['visit_id']
print(np.max(visits["day_obs"][selection].value_counts()))

In [ ]:
# ORIGINAL
observer = Observer.at_site("LSST")

In [ ]:
# ORIGINAL
def dayObsToTime(day_obs):
    year = str(day_obs)[0:4]
    month = str(day_obs)[4:6]
    day = str(day_obs)[6:8]
    time = Time("%s-%s-%s 00:00:00" % (year, month, day), format="iso")
    return time

In [ ]:
# ORIGINAL
def dayObsToMidnight(day_obs, observer):
    year = str(day_obs)[0:4]
    month = str(day_obs)[4:6]
    day = str(day_obs)[6:8]
    time = Time("%s-%s-%s 23:59:00" % (year, month, day), scale="utc", format="iso")

    time_midnight = observer.midnight(time, which="nearest")
    return time_midnight

In [ ]:
# ORIGINAL
day_obs_min_time_midnight = dayObsToMidnight(day_obs_min, observer)

offsets = np.arange(
    -1,
    np.floor(
        1.5
        + (dayObsToMidnight(day_obs_max, observer) - day_obs_min_time_midnight)
        .to(u.day)
        .value
    )
    + 1,
)

time_day_obs_twilight_plotting = dayObsToTime(day_obs_min) + (offsets * u.day)
time_evening_astronomical_twilight = observer.twilight_evening_astronomical(
    day_obs_min_time_midnight + offsets * u.day, which="previous"
)
time_morning_astronomical_twilight = observer.twilight_morning_astronomical(
    day_obs_min_time_midnight + offsets * u.day, which="next"
)
time_evening_nautical_twilight = observer.twilight_evening_nautical(
    day_obs_min_time_midnight + offsets * u.day, which="previous"
)
time_morning_nautical_twilight = observer.twilight_morning_nautical(
    day_obs_min_time_midnight + offsets * u.day, which="next"
)

In [ ]:
# ORIGINAL
def convertTime(time_input, time_reference):
    time_output = (time_input - (time_reference + (1.0 * u.day))) * u.day.to(u.hr)
    return time_output.value

In [ ]:
# ORIGINAL
time_exp_midpt_mjd = Time(visits["exp_midpt_mjd"], format="mjd")
time_obs_start_mjd = Time(visits["obs_start_mjd"], format="mjd")
time_obs_end_mjd = Time(visits["obs_end_mjd"], format="mjd")

## Visit Timeline

ORIGINAL markdown follows. Note that it says "relative to local midnight"; the code (and the
RTN-122 caption) actually reference **00:00 UTC** on the calendar day after `day_obs`.

> Each column is one night; each tick marks a visit at its time relative to local
> midnight (y-axis, hours). The grey bands show evening/morning twilight (nautical
> −12° and astronomical −18°), so ticks outside the bands are dark-time observing.
> Visits are colored by program class: survey programs (blue), engineering (orange),
> and focus & alignment (green). Reading down a column shows how fully each night was
> used; reading across shows night-to-night consistency and seasonal change in night
> length.

In [ ]:
# ORIGINAL (only the savefig path is modified, at the bottom)
plt.figure(figsize=(16, 8), dpi=200)

xticks = np.array(
    [dayObsToTime(day_obs).mjd for day_obs in np.unique(visits["day_obs"])]
)
xtick_labels = np.unique(visits["day_obs"])

day_obs_summary = {
    "day_obs": xtick_labels,
    "day_obs_mjd": xticks,
}
for key in day_obs_summary.keys():
    day_obs_summary[key] = np.array([_ for _ in day_obs_summary[key]])

# Twilight plotting
fill_high = np.tile(16, len(time_day_obs_twilight_plotting))
fill_low = np.tile(-4, len(time_day_obs_twilight_plotting))
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    convertTime(time_morning_astronomical_twilight, time_day_obs_twilight_plotting),
    convertTime(time_morning_nautical_twilight, time_day_obs_twilight_plotting),
    alpha=0.2,
    color="black",
    label="-12 deg to -18 deg Twilight",
)
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    convertTime(time_morning_nautical_twilight, time_day_obs_twilight_plotting),
    fill_high,
    alpha=0.4,
    color="black",
)
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    convertTime(time_evening_nautical_twilight, time_day_obs_twilight_plotting),
    convertTime(time_evening_astronomical_twilight, time_day_obs_twilight_plotting),
    alpha=0.2,
    color="black",
)
plt.fill_between(
    time_day_obs_twilight_plotting.mjd,
    fill_low,
    convertTime(time_evening_nautical_twilight, time_day_obs_twilight_plotting),
    alpha=0.4,
    color="black",
)

x_engineering = []
y_engineering = []
x_survey = []
y_survey = []
x_focus_alignment = []
y_focus_alignment = []
for day_obs in np.unique(visits["day_obs"]):

    time_day_obs = dayObsToTime(day_obs)

    selection_day_obs = visits["day_obs"] == day_obs

    time_exp_midpt_mjd = convertTime(
        Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), time_day_obs
    )
    time_obs_start_mjd = convertTime(
        Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), time_day_obs
    )
    time_obs_end_mjd = convertTime(
        Time(visits["exp_midpt_mjd"][selection_day_obs], format="mjd"), time_day_obs
    )
    x = np.tile(time_day_obs.mjd, len(visits[selection_day_obs]))

    selection_fbs = np.isin(
        visits[selection_day_obs]["science_program"], SURVEY_PROGRAMS
    )
    selection_aos = np.isin(
        visits[selection_day_obs]["science_program"], FOCUS_ALIGNMENT_PROGRAMS
    )

    x_engineering.append(x[~selection_fbs])
    y_engineering.append(time_exp_midpt_mjd[~selection_fbs])
    x_survey.append(x[selection_fbs])
    y_survey.append(time_exp_midpt_mjd[selection_fbs])
    x_focus_alignment.append(x[selection_aos])
    y_focus_alignment.append(time_exp_midpt_mjd[selection_aos])

# All scatter plots at the end
s = 20
plt.scatter(
    np.concatenate(x_engineering),
    np.concatenate(y_engineering),
    marker="_",
    s=s,
    c="tab:orange",
    label="Engineering",
)
# plt.scatter(np.concatenate(x_survey), np.concatenate(y_survey), marker='_', s=s, c='tab:blue', label='Pre-LSST')
plt.scatter(
    np.concatenate(x_survey),
    np.concatenate(y_survey),
    marker="_",
    s=s,
    c="tab:blue",
    label="Survey Programs",
)
plt.scatter(
    np.concatenate(x_focus_alignment),
    np.concatenate(y_focus_alignment),
    marker="_",
    s=s,
    c="tab:green",
    label="Focus and Alignment",
)

plt.legend(loc="upper left", markerscale=2)

plt.xlim(xticks[0] - 1.0, xticks[-1] + 1.0)
plt.ylim(-1.5, 11.0)  # Early Ops
# plt.ylim(-2., 11.) # Commissioning
plt.xticks(xticks, xtick_labels, rotation=90.0, fontsize=8)
plt.ylabel("Time (hrs)")
plt.minorticks_off()
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(PLOTS_DIR / ("on_sky_utilization_repro_%i-%i.png" % (day_obs_min, day_obs_max)))
    plt.savefig(PLOTS_DIR / ("on_sky_utilization_repro_%i-%i.pdf" % (day_obs_min, day_obs_max)))
plt.show()

## Bookkeeping (added)

Counts that help compare against the screenshot and the interactive version. The "drawn twice"
number is the size of the orange-under-green overlap caused by Engineering being defined as
"not survey".

In [ ]:
# ADDED: numbers for the comparison
is_survey = np.isin(visits["science_program"], SURVEY_PROGRAMS)
is_focus = np.isin(visits["science_program"], FOCUS_ALIGNMENT_PROGRAMS)
summary = pd.Series(
    {
        "visits plotted": len(visits),
        "nights with visits": visits["day_obs"].nunique(),
        "calendar nights in range": int(
            (dayObsToTime(day_obs_max) - dayObsToTime(day_obs_min)).to(u.day).value
        )
        + 1,
        "survey (blue)": int(is_survey.sum()),
        "focus and alignment (green)": int(is_focus.sum()),
        "engineering (orange, = not survey)": int((~is_survey).sum()),
        "drawn twice (orange and green)": int((is_focus & ~is_survey).sum()),
        "y range of data (hrs)": "%.2f to %.2f"
        % (
            np.concatenate(y_engineering + y_survey).min(),
            np.concatenate(y_engineering + y_survey).max(),
        ),
    }
)
print(summary.to_string())